In [1]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

# Standard library imports
import argparse
import importlib.util
import inspect
import json
import math
import os
import pickle
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import warnings
from functools import reduce

# Third-party imports
import librosa
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from matplotlib import cm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import umap
from pathlib import Path
import cv2

from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Local imports
import commons
import lightning_wrapper
import losses
import models
import utils
import train
from cough_datasets import (
    CoughDatasets,
    CoughDatasetsCollate,
    CoughDatasetsProcessorCollate,
    CoughDetectionRatioBatchSampler,
    CoughDiseaseBinaryBatchSampler,
    PatientBatchSampler
)

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

def find_bilstmatt_logmel_logs(root="."):
    root = Path(root)
    log_folders = set()

    for p in root.rglob("bilstmatt_logmel"):
        if p.is_dir():
            parent = p.parent
            if parent.name == "logs" or parent.name.startswith("logs_"):
                log_folders.add(parent)

    return sorted(str(p) for p in log_folders)

class CAMWrapper(torch.nn.Module):
    def __init__(self, lightning_model):
        super().__init__()
        self.lightning_model = lightning_model

    def forward(self, x):
        out = self.lightning_model(x)
        return out["disease_logits"]

class SplitStreamCAMWrapper(nn.Module):
    def __init__(self, model, stream_idx: int):
        super().__init__()
        self.model = model
        self.stream_idx = stream_idx  # 0=raw, 1=delta, 2=deltadelta

    def forward(self, s):
        z = self.model.forward_encoder(s.unsqueeze(1))
        logits = self.model.classifier(z)
        return logits

def minmax_norm(x, eps=1e-8):
    return (x - x.min()) / (x.max() - x.min() + eps)

def cosine_sim(x, y, axis):
    num = np.sum(x * y, axis=axis)
    den = np.linalg.norm(x, axis=axis) * np.linalg.norm(y, axis=axis)
    return num / (den + 1e-8)

def unified_confidence(p, tau=0.4484136):
    """
    Returns signed confidence in range [-100, +100].
    Positive = TB
    Negative = Non-TB
    0 = exactly at decision threshold
    """
    if p >= tau:
        # TB side
        return 100.0 * (p - tau) / (1.0 - tau)
    else:
        # Non-TB side
        return -100.0 * (tau - p) / tau

def reduce_embeddings(embeddings, method="umap", random_state=42):
    if method == "umap":
        reducer = umap.UMAP(
            n_neighbors=15,
            min_dist=0.1,
            n_components=2,
            random_state=random_state
        )
    elif method == "tsne":
        reducer = TSNE(
            n_components=2,
            perplexity=30,
            learning_rate="auto",
            init="pca",
            random_state=random_state
        )
    else:
        raise ValueError("method must be 'umap' or 'tsne'")
    
    return reducer.fit_transform(embeddings)


def scatter_plot(Z, color, title, cmap="viridis", cbar_label=None):
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z[:, 0], Z[:, 1], c=color, cmap=cmap, s=12, alpha=0.8)
    plt.title(title)
    plt.xticks([])
    plt.yticks([])
    if cbar_label is not None:
        cbar = plt.colorbar(sc)
        cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.show()

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


/tmp/ipykernel_950380/775526060.py:66: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis")


In [ ]:
#now_experiment = "logs/resnet34_logmel"
use_cpu = False

for now_experiment in os.listdir("./logs"): # ['resnet34_melspectogram']
    now_experiment = f"logs/{now_experiment}"
    experiment_metadata = now_experiment.split("/")
    parser = train.parse_args()
    args = parser.parse_args(["--model_name", experiment_metadata[1]])

    model_dir = os.path.join(experiment_metadata[0], args.model_name)
    config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
    hps = train.load_config(config_path, model_dir, args)

    # =============================================================
    # SECTION: Loading Data
    # =============================================================
    df_train, df_test = train.load_data(hps)
    collate_fn = train.get_collate_fn(hps)
    target_labels = df_train[hps.data.target_column]

    logger = utils.get_logger(hps.model_dir, filename="dummy.log")
    logger.info(hps)

    pool_net, pool_model = train.setup_model(hps, is_init=args.init)
    runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
        os.path.join(hps.model_dir, "best_model.ckpt"),
        model=pool_model,
        hps=hps, custom_logger=logger
    )
    runner_lightning.eval()
    trainer = L.Trainer(accelerator="gpu" if use_cpu == False else "cpu", devices="auto")

    info_fold_data = train.load_fold_info(hps.model_dir)
    best_fold_idx = info_fold_data.get("best_fold_idx", 0)

    splitter, num_folds = train.create_data_split(df_train, target_labels, use_kfold=hps.train.use_Kfold)
    train_idx, val_idx = list(splitter)[best_fold_idx]

    train_fold = df_train.iloc[train_idx].reset_index(drop=True)
    val_fold = df_train.iloc[val_idx].reset_index(drop=True)
    train_loader, val_loader = train.prepare_fold_data(train_fold, val_fold, hps, best_fold_idx, collate_fn)

    test_probs = []
    test_labels = []
    with torch.no_grad():
        for idx, batch in tqdm(enumerate(train_loader), total=len(train_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
            audio = audio.cuda()
            
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            probs = torch.sigmoid(out_model['disease_logits']).squeeze(-1)
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            test_labels.extend(labels)
            test_probs.extend(probs.cpu().detach().numpy())

    optimized_threshold = utils.optimize_threshold_youden(test_labels, test_probs)
    # fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
    # spec = 1 - fpr

    # sens_v = np.maximum(0.80 - tpr, 0.0)
    # spec_v = np.maximum(0.60 - spec, 0.0)

    # # primary = sens_v + spec_v
    # # secondary = (tpr - 0.80)**2 + (spec - 0.60)**2
    # # loss = primary * 1000.0 + secondary
    # # best_threshold = thresholds[np.argmin(loss)]

    # loss = (
    #     10000.0 * sens_v +
    #     100.0 * spec_v +
    #     (tpr - 0.80)**2 +
    #     (spec - 0.60)**2
    # )

    # best_threshold = thresholds[np.argmin(loss)]
    #runner_lightning.probs_threshold = best_threshold #optimized_threshold
    runner_lightning.probs_threshold = optimized_threshold

    # ############################################# Overall Report #########################################################
    # with open(f"{model_dir}/result_overall.txt", "w") as f:
    #     f.write(f"")

    # df_train_eval = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
    # df_train_eval = df_train_eval.reset_index(drop=True)
    # df_train_eval = df_train_eval[hps.data.column_order]

    # results_dict = train.evaluate_on_dataset(runner_lightning, trainer, df_train_eval, hps, best_fold_idx, collate_fn)
    # train.write_results_to_file("result_overall.txt", results_dict, model_dir, {0: "Train"})

    # df_test_eval = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')
    # df_test_eval = df_test_eval.reset_index(drop=True)
    # df_test_eval = df_test_eval[hps.data.column_order]

    # runner_lightning.generate_figure = True
    # results_dict = train.evaluate_on_dataset(runner_lightning, trainer, df_test_eval, hps, best_fold_idx, collate_fn)
    # train.write_results_to_file("result_overall.txt", results_dict, model_dir, {0: "Test"})
    # runner_lightning.generate_figure = False

    # df_cirdz = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cirdz.csv.test')
    # df_cirdz = df_cirdz.reset_index(drop=True)
    # df_cirdz = df_cirdz[hps.data.column_order]

    # results_dict = train.evaluate_on_dataset(runner_lightning, trainer, df_cirdz, hps, best_fold_idx, collate_fn)
    # train.write_results_to_file("result_overall.txt", results_dict, model_dir, {0: "Unseen"})

INFO:cnnbilstm_logmel:{'train': {'use_cuda': True, 'log_interval': 20, 'seed': 1234, 'epochs': 10000, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'betas': [0.8, 0.99], 'eps': 1e-09, 'lr_decay': 0.999875, 'batch_size': 128, 'loss_function': 'BCE', 'use_Kfold': True, 'mae_training': False, 'ssccl_training': False}, 'data': {'max_wav_value': False, 'mean_std_norm': True, 'per_band_norm': False, 'sampling_rate': 16000, 'filter_length': 2048, 'hop_length': 256, 'win_length': 512, 'n_mel_channels': 80, 'mel_fmin': 50.0, 'mel_fmax': 8000.0, 'saming_length': True, 'desired_length': 0.55, 'fade_samples_ratio': 16, 'pad_types': 'zero', 'rezize_size': [64, 256], 'tabular_feature': True, 'acoustic_feature': True, 'feature_type': 'logmel', 'delta_feature': False, 'deltadelta_feature': False, 'multimask_augment': False, 'multimask_prob': 0.4, 'tau': 0.1, 'nu': 0.1, 'num_masks': 4, 'augment_data': True, 'augment_prob': 0.5, 'augment_rawboost': True, 'add_noise': False, 'cough_detection': False, 

INFO:cnnbilstm_logmel:Trainable params: 1663105 | Total params: 1663105 | Trainable%: 100.00% | Size: 1.66M


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
100%|██████████| 45/45 [00:16<00:00,  2.65it/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6835694313049316     │
│        test_auroc         │    0.7269701361656189     │
│         test_bacc         │    0.6709259152412415     │
│        test_pauroc        │            0.0            │
│         test_sens         │    0.6332622766494751     │
│         test_spec         │    0.7085896134376526     │
└───────────────────────────┴───────────────────────────┘

...

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6751958131790161     │
│        test_auroc         │    0.7190670371055603     │
│         test_bacc         │    0.6585695743560791     │
│        test_pauroc        │            0.0            │
│         test_sens         │     0.596017062664032     │
│         test_spec         │    0.7211220860481262     │
└───────────────────────────┴───────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.31929823756217957    │
│        test_auroc         │    0.6675129532814026     │
│         test_bacc         │    0.5773208737373352     │
│        test_pauroc        │            0.0            │
│         test_sens         │    0.9650349617004395     │
│         test_spec         │    0.18960674107074738    │
└───────────────────────────┴───────────────────────────┘

In [4]:
runner_lightning.probs_threshold

0.42902732

In [5]:
count_1 = test_labels.count(1)
count_0 = test_labels.count(0)


In [7]:
count_0

2817

In [6]:
count_1

2838

In [22]:
spec_v

array([0.        , 0.        , 0.        , ..., 0.57720098, 0.57720098,
       0.6       ])

In [17]:
probs

tensor([0.1653, 0.2789, 0.7529, 0.0877, 0.1325, 0.4862, 0.1042, 0.0314, 0.5980,
        0.2283, 0.3853, 0.2652, 0.7762, 0.0506, 0.1577, 0.4660, 0.5591, 0.1735,
        0.1311, 0.4367, 0.1610, 0.7180, 0.5117, 0.0205, 0.3055, 0.6969, 0.8587,
        0.2816, 0.7690], device='cuda:0')

In [19]:
valid = []
for thr in thresholds:
    preds = (probs.cpu().numpy() >= thr).astype(int)
    TN, FP, FN, TP = confusion_matrix(labels, preds).ravel()
    sens = TP / (TP + FN) if TP + FN else 0
    spec = TN / (TN + FP) if TN + FP else 0
    valid.append((thr, sens, spec))

valid = np.array(valid, dtype=object)

# thresholds that truly satisfy constraints
feasible = valid[(valid[:,1] >= 0.80) & (valid[:,2] >= 0.60)]
print("Number of realizable feasible thresholds:", len(feasible))


Number of realizable feasible thresholds: 0


In [15]:
dist

array([0.8       , 0.79944029, 0.79107403, ..., 0.38461306, 0.38475289,
       0.4       ])

In [10]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
spec = 1 - fpr

min_tpr = 0.80
min_spec = 0.60
mask = (tpr >= min_tpr) & (spec >= min_spec)

if np.any(mask):
    youden_j = tpr - fpr
    best_idx = np.argmax(youden_j[mask])
    best_threshold = thresholds[mask][best_idx]
else:
    # fallback: closest point to constraints
    penalty = (np.maximum(0, min_tpr - tpr) ** 2 +
               np.maximum(0, min_spec - spec) ** 2)
    best_threshold = thresholds[np.argmin(penalty)]

In [11]:
best_threshold

0.26538184

In [8]:
mask.sum()

5

In [ ]:
balance_score = np.abs(tpr - spec)

In [ ]:
balance_score

In [ ]:
mask.sum()

In [ ]:
test_probs

In [ ]:


with open(os.path.join(model_dir, "probs_threshold.pkl"), "rb") as f:
    runner_lightning.probs_threshold = pickle.load(f)['probs_threshold']

info_fold_data = train.load_fold_info(hps.model_dir)


train_dataset = CoughDatasets(
    df_train.values, 
    hps.data,
    wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
    train=False
)
test_dataset = CoughDatasets(
    df_test.values, 
    hps.data,
    wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
    train=False
)

# Create sampler
#sampler = train.create_sampler(train_fold, hps)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    num_workers=28, 
    #sampler=sampler, 
    batch_size=hps.train.batch_size,
    pin_memory=True, 
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, 
    num_workers=28, 
    shuffle=False, 
    batch_size=hps.train.batch_size,
    pin_memory=True, 
    collate_fn=collate_fn
)
